# Домашнє завдання: Прогнозування орендної плати за житло

## Мета завдання
Застосувати знання з лекції для побудови моделі лінійної регресії, що прогнозує орендну плату за житло в Індії. Ви пройдете весь цикл вирішення задачі машинного навчання: від дослідницького аналізу до оцінки якості моделі.

## Опис датасету
**House Rent Prediction Dataset** містить інформацію про 4700+ оголошень про оренду житла в Індії з такими параметрами:
- **BHK**: Кількість спалень, залів, кухонь
- **Rent**: Орендна плата (цільова змінна)
- **Size**: Площа в квадратних футах
- **Floor**: Поверх та загальна кількість поверхів
- **Area Type**: Тип розрахунку площі
- **Area Locality**: Район
- **City**: Місто
- **Furnishing Status**: Стан меблювання
- **Tenant Preferred**: Тип орендаря
- **Bathroom**: Кількість ванних кімнат
- **Point of Contact**: Контактна особа

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `House_Rent_Dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (включно з типами даних та кількістю значень)


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = ('/content/drive/MyDrive/Data Analyst/ML/House_Rent_Dataset.csv')
df = pd.read_csv(data_path)

In [ ]:
df.shape

(4746, 12)

In [ ]:
df.head()

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4746 entries, 0 to 4745
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Posted On          4746 non-null   object
 1   BHK                4746 non-null   int64 
 2   Rent               4746 non-null   int64 
 3   Size               4746 non-null   int64 
 4   Floor              4746 non-null   object
 5   Area Type          4746 non-null   object
 6   Area Locality      4746 non-null   object
 7   City               4746 non-null   object
 8   Furnishing Status  4746 non-null   object
 9   Tenant Preferred   4746 non-null   object
 10  Bathroom           4746 non-null   int64 
 11  Point of Contact   4746 non-null   object
dtypes: int64(4), object(8)
memory usage: 445.1+ KB


## Завдання 2: Дослідницький аналіз даних (EDA) (5 балів)

**Що потрібно зробити:**
1. **Аналіз пропущених значень.** Перевірте наявність і відсоток пропущених значень у кожній колонці
2. **Базова статистика.** Обчисліть базову статистику (середнє, квартилі, стандартне відхилення) для числових змінних.
3. **Аналіз цільової змінної.** Побудуйте гістограму розподілу цільової змінної (Rent)
4. **Робота з викидами.** Знайдіть та видаліть викиди в цільовій змінній (якщо є). Визначити викиди можна будь-яким зрозумілим для вас способом, як варіант - таким, що використовується в побудові box-plot (https://en.wikipedia.org/wiki/Box_plot#Example_with_outliers).
5. **Аналіз категоріальних змінних.** Виведіть кількість унікальних значень для кожної з категоріальних колонок.


In [ ]:
# 1. Аналіз пропущених значень у кожній колонці
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100

missing_percent

,0
Posted On,0.0
BHK,0.0
Rent,0.0
Size,0.0
Floor,0.0
Area Type,0.0
Area Locality,0.0
City,0.0
Furnishing Status,0.0
Tenant Preferred,0.0


In [ ]:
# 2. Базова статистика для числових змінних
stats = df[['BHK', 'Rent', 'Size', 'Bathroom']].describe()
stats.round(2)

,BHK,Rent,Size,Bathroom
count,4746.00,4746.00,4746.00,4746.00
mean,2.08,34993.45,967.49,1.97
std,0.83,78106.41,634.20,0.88
min,1.00,1200.00,10.00,1.00
25%,2.00,10000.00,550.00,1.00
50%,2.00,16000.00,850.00,2.00
75%,3.00,33000.00,1200.00,2.00
max,6.00,3500000.00,8000.00,10.00


Короткий опис переглянутої базової статистики:
1. Пропусків в датасеті немає.          
2. Rent:
    - Середнє значення близко 34993.45, проте стандартне відхилення 78106.41 - воно більш ніж удвічі перевищує середнє. Це дає нам зрозуміти, що ми маємо сильний правосторонній скос та також присутні викиди.
    - Медіана (50%) - 16000.00, а макимальне значення (max) - 3500000.00. Тобто ми бачимо, що медіанна майже двічі менше за середнє (розподіл тягнеться в право).
3. Size:
   - Подібна ситуація до Rent, середнє значення - 967.49, медіана - 850.00. Max = 8000.00, а ось min = 10.00 - це або аномалія або помилка в даних, варто буде перевірити.

In [ ]:
# 3. Аналіз цільової змінної - Rent

fig = px.histogram(
    df,
    x='Rent',
    nbins=100,
    title='Розподіл цільової змінної (орендна плата)',
    labels={'Rent': 'Орендна плата', 'count': 'Кількість оголошень'}
)
fig.update_layout(
    showlegend=False,
    height=400
)

fig.update_yaxes(title_text='Кількість оголошень')
fig.update_xaxes(title_text='Орендна плата')

fig.show()

In [ ]:
# 4. Робота з викидами - Rent

# Візуальна оцінка розподілу до очищення
fig_before = px.box(
    df,
    y='Rent',
    title='Розподіл орендної плати (до очищення)',
    labels={'Rent': 'Орендна плата'},
    color_discrete_sequence=['#00CC96']
)
fig_before.update_layout(height=450)
fig_before.show()

In [ ]:
# Розрахунок меж за методом IQR
Q1, Q3 = df['Rent'].quantile([0.25, 0.75])
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR
# Нижня межа (Q1 - 1.5 * IQR) = -24500 — не застосовується,
# оскільки орендна плата не може бути від'ємною

print(f'Q1 = {Q1:,.0f} | Q3 = {Q3:,.0f} | IQR = {IQR:,.0f}')
print(f'Верхня межа викидів: {upper_limit:,.0f}')

Q1 = 10,000 | Q3 = 33,000 | IQR = 23,000
Верхня межа викидів: 67,500


In [31]:
# Розділяємо дані на "нормальні" та викиди
mask_outliers = df['Rent'] > upper_limit
outliers = df[mask_outliers]
df_clean = df[~mask_outliers].copy()

print(f'\nВикидів знайдено: {len(outliers)} ({len(outliers) / len(df) * 100:.2f}% датасету)')
print(f'Датасет: {len(df)} → {len(df_clean)} записів')


Викидів знайдено: 520 (10.96% датасету)
Датасет: 4746 → 4226 записів


In [ ]:
# Дивимось на топ-10 найдорожчих викидів
print('\nТоп-10 викидів:')
outliers.nlargest(10, 'Rent')[['Rent', 'BHK', 'Size', 'City', 'Area Locality']]


Топ-10 викидів:


,Rent,BHK,Size,City,Area Locality
1837,3500000,3,2500,Bangalore,Marathahalli
1001,1200000,4,5000,Mumbai,Juhu
827,1000000,4,3064,Mumbai,"Raheja Artesia, Worli"
1329,850000,4,3200,Mumbai,Breach Candy
1459,700000,4,3200,Mumbai,"Lady Ratan Tower, Worli"
1484,680000,4,1962,Mumbai,Khar West
1319,650000,5,3000,Mumbai,Khar West
726,600000,4,2500,Mumbai,"Mount Marry, Bandra West"
792,600000,5,3200,Mumbai,Bandra East
1384,600000,5,4500,Mumbai,Bandra West


In [ ]:
# Візуальна перевірка після очищення
fig_after = px.box(
    df_clean, y='Rent',
    title='Розподіл орендної плати (після очищення)',
    labels={'Rent': 'Орендна плата'},
    color_discrete_sequence=['#00CC96']
)
fig_after.update_layout(height=450)
fig_after.show()

Застосований метод(IQR) для виявлення викидів у цільовій змінній Rent.          

Розрахована верхня межа становить 67 500 — усі значення вище неї вважаються викидами. Нижня межа (-24 500) не застосовувалась, оскільки орендна плата не може бути від'ємною.

Було виявлено 520 викидів (10.96% датасету). Після їх видалення датасет скоротився з 4746 до 4226 записів.

Метод IQR був обраний як стійкий до екстремальних значень, на відміну від підходу на основі середнього та стандартного відхилення.

In [34]:
# 5. Аналіз категоріальних змінних
cat_cols = df_clean.select_dtypes(include='object').columns
unique_counts = df_clean[cat_cols].nunique()

print('Кількість унікальних значень у категоріальних колонках:')
print(unique_counts)

Кількість унікальних значень у категоріальних колонках:
Posted On              80
Floor                 340
Area Type               3
Area Locality        1997
City                    6
Furnishing Status       3
Tenant Preferred        3
Point of Contact        3
dtype: int64



## Завдання 3: Аналіз кореляцій та взаємозв'язків (3 бали)

**Що потрібно зробити:**
1. Обчисліть матрицю кореляцій для числових змінних
2. Візуалізуйте кореляційну матрицю за допомогою heatmap
3. Побудуйте scatter plot між Size та Rent
4. Проаналізуйте взаємозв'язок між BHK та Rent за допомогою boxplot (який розподіл плати для різних значень BHK)


In [35]:
# 1. Матриця кореляцій для числових змінни

# Створюємо датафрейм тільки з числовими метриками
metrics_df = df_clean[['BHK', 'Rent', 'Size', 'Bathroom']].dropna()

# Матриця кореляцій
correlation_matrix = metrics_df.corr()

In [36]:
# 2. Візуалізація кореляційної матриці

fig = px.imshow(
    correlation_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Кореляція між числовими змінними',
    labels=dict(color="Кореляція")
)
fig.update_layout(height=500)
fig.show()

Найбільший зв'язок із Rent має Bathroom (0.51), за нею BHK (0.40) та Size (0.39) — помірні позитивні кореляції, що означають зростання орендної плати зі збільшенням цих параметрів.         

Між самими ознаками кореляції сильніші (BHK–Bathroom = 0.75, BHK–Size = 0.70).      

Оскільки числові змінні пояснюють Rent лише частково, для якісного прогнозування важливо враховувати і категоріальні ознаки (місто, район, тип меблювання).

In [39]:
# 3. Scatter plot між Size та Rent

fig = px.scatter(
    df_clean,
    x='Size',
    y='Rent',
    title='Звʼязок між площею житла та орендною платою',
    labels={
        'Size': 'Площа житла',
        'Rent': 'Орендна плата'
    },
    opacity=0.5,
    hover_data=['City', 'BHK', 'Bathroom']
)

fig.update_layout(height=500)
fig.show()

Scatter plot підтверджує помірний позитивний зв'язок між Size та Rent (кореляція 0.39): зі збільшенням площі орендна плата зростає, але точки значно розсіяні. Це означає, що площа сама по собі не пояснює ціну повністю — значну роль відіграють інші фактори (місто, район, меблювання, кількість кімнат).

In [40]:
# 4. Взаємозв'язок між BHK та Rent за допомогою boxplot

fig = px.box(
    df_clean,
    x='BHK',
    y='Rent',
    color='BHK',
    title='Розподіл орендної плати залежно від к-сті кімнат (BHK)',
    labels={'BHK': 'Кількість кімнат', 'Rent': 'Орендна плата'}
)
fig.update_layout(height=500, showlegend=False)
fig.show()

Boxplot демонструє позитивну залежність між BHK та Rent: медіана і розкид орендної плати зростають разом із кількістю кімнат, особливо помітний стрибок від 2 до 3 BHK. При цьому у групах 1–2 BHK спостерігаються поодинокі високі викиди — ймовірно, це житло у престижних районах або з покращеними характеристиками. Це підтверджує, що BHK впливає на вартість оренди, проте не є єдиним визначальним фактором.

## Завдання 4: Feature Engineering та підготовка даних (4 бали)

**Що потрібно зробити:**
1. Закодуйте категоріальні змінні за допомогою One-Hot Encoding. Пригадайте, що в лекції ми говорили щодо кодування кат. змінних з великої кількістю різних значень і як працювати з такими випадками. Ви можете закодувати не всі кат. змінні, а лише ті, що вважаєте за потрібні (скажімо ті, що мають відносно небагато різних значень).
2. **Опціонально (по 0.5 бала за кожну доцільну ознаку):** Додайте нові ознаки, обчислені на основі наявних даних, які б на ваш погляд були корисними для моделі
3. Виберіть ознаки для побудови моделі (виключіть непотрібні колонки). Виключити можна, наприклад, ті колонки, які мають категоріальний тип і забагато (більше 20) різних значень. Треба виключити хоча б 1 колонку.
4. Розділіть дані на ознаки (X) та цільову змінну (y)
5. Застосуйте стандартизацію до числових ознак


In [42]:
# 1. Копія очищеного датасету

df_model = df_clean.copy()

In [43]:
# Перевіряємо к-сть унікальних значень у категоріальних колонках
df_model.select_dtypes(include='object').nunique().sort_values(ascending=False)

,0
Area Locality,1997
Floor,340
Posted On,80
City,6
Area Type,3
Furnishing Status,3
Tenant Preferred,3
Point of Contact,3


In [44]:
# 2. Feature Engineering

# Площа на одну кімнату — допомагає відрізнити просторі квартири від тісних
df_model['Size_per_BHK'] = df_model['Size'] / df_model['BHK']

# К-сть ванних на кімнату — індикатор преміальності житла
df_model['Bathroom_per_BHK'] = df_model['Bathroom'] / df_model['BHK']

# Місяць публікації — може відображати сезонність попиту
df_model['Posted On'] = pd.to_datetime(df_model['Posted On'])
df_model['Posted_Month'] = df_model['Posted On'].dt.month

In [47]:
# 3. Відбір ознак та One-Hot Encoding

# Числові ознаки
numeric_cols = [
    'BHK', 'Size', 'Bathroom',
    'Size_per_BHK', 'Bathroom_per_BHK', 'Posted_Month'
]

# Категоріальні з невеликою к-стю унікальних значень
# Виключено: Area Locality (2233 значень), Floor (490 значень), Posted On (дата)
categorical_cols = [
    'Area Type',
    'City',
    'Furnishing Status',
    'Tenant Preferred',
    'Point of Contact'
]

In [48]:
# One-Hot Encoding з видаленням першої категорії для уникнення мультиколінеарності
df_encoded = pd.get_dummies(
    df_model[numeric_cols + categorical_cols],
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print(f'Розмір датасету після кодування: {df_encoded.shape}')
df_encoded.head()

Розмір датасету після кодування: (4226, 19)


,BHK,Size,Bathroom,Size_per_BHK,Bathroom_per_BHK,Posted_Month,Area Type_Carpet Area,Area Type_Super Area,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Point of Contact_Contact Builder,Point of Contact_Contact Owner
0,2,1100,2,550.0,1.0,5,0,1,0,0,0,1,0,0,1,1,0,0,1
1,2,800,1,400.0,0.5,5,0,1,0,0,0,1,0,1,0,1,0,0,1
2,2,1000,1,500.0,0.5,5,0,1,0,0,0,1,0,1,0,1,0,0,1
3,2,800,1,400.0,0.5,7,0,1,0,0,0,1,0,0,1,1,0,0,1
4,2,850,1,425.0,0.5,5,1,0,0,0,0,1,0,0,1,0,0,0,1


In [49]:
# 4. Розділення на ознаки (X) та цільову змінну (y)

X = df_encoded
y = df_model['Rent']

print(f'Розмір X: {X.shape}')
print(f'Розмір y: {y.shape}')

Розмір X: (4226, 19)
Розмір y: (4226,)


In [51]:
# 5. Стандартизація числових ознак

scaler = StandardScaler()

X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

X.head()

,BHK,Size,Bathroom,Size_per_BHK,Bathroom_per_BHK,Posted_Month,Area Type_Carpet Area,Area Type_Super Area,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,Furnishing Status_Semi-Furnished,Furnishing Status_Unfurnished,Tenant Preferred_Bachelors/Family,Tenant Preferred_Family,Point of Contact_Contact Builder,Point of Contact_Contact Owner
0,0.052966,0.469859,0.272578,0.604132,0.147328,-0.861071,0,1,0,0,0,1,0,0,1,1,0,0,1
1,0.052966,-0.147778,-1.133910,-0.268747,-1.637671,-0.861071,0,1,0,0,0,1,0,1,0,1,0,0,1
2,0.052966,0.263980,-1.133910,0.313173,-1.637671,-0.861071,0,1,0,0,0,1,0,1,0,1,0,0,1
3,0.052966,-0.147778,-1.133910,-0.268747,-1.637671,1.546396,0,1,0,0,0,1,0,0,1,1,0,0,1
4,0.052966,-0.044839,-1.133910,-0.123267,-1.637671,-0.861071,1,0,0,0,0,1,0,0,1,0,0,0,1


## Завдання 5: Розділення даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на навчальну (80%) та тестову (20%) вибірки.
2. Створіть модель лінійної регресії.
3. Навчіть модель на навчальних даних.
4. Виведіть усі коефіцієнти моделі (ваги) та напишіть, які 2 ознаки найбільше впливають на прогноз.
5. Зробіть прогнози на тренувальній та тестовій вибірках.

In [52]:
# 1. Розділяємо дані: 80% на навчання, 20% на тест

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,                                    # 20% даних йде на тест
    random_state=42                                   # фіксуємо випадковість для відтворюваності
)

In [53]:
# 2. Створюємо модель
model = LinearRegression()

In [54]:
# 3. Навчаємо модель на навчальних даних
model.fit(X_train, y_train)

LinearRegression()

In [55]:
# 4. Виводимо коефіцієнти моделі
coef_df = pd.DataFrame({
    'Ознака': X_train.columns,
    'Коефіцієнт': model.coef_.round(2)
})

coef_df['|Коефіцієнт|'] = coef_df['Коефіцієнт'].abs()
coef_df = coef_df.sort_values('|Коефіцієнт|', ascending=False)

print(f'Intercept (зміщення): {model.intercept_:.2f}\n')
coef_df.head(10)

Intercept (зміщення): 30466.70



,Ознака,Коефіцієнт,|Коефіцієнт|
12,City_Mumbai,19210.40,19210.40
18,Point of Contact_Contact Owner,-8597.11,8597.11
17,Point of Contact_Contact Builder,-5644.63,5644.63
14,Furnishing Status_Unfurnished,-4595.11,4595.11
2,Bathroom,4262.12,4262.12
13,Furnishing Status_Semi-Furnished,-3431.68,3431.68
1,Size,3103.92,3103.92
11,City_Kolkata,-2999.52,2999.52
7,Area Type_Super Area,-2485.38,2485.38
10,City_Hyderabad,-2432.73,2432.73


Найбільший вплив на прогноз орендної плати мають дві ознаки:
- City_Mumbai (коефіцієнт +19 210)
- Point of Contact_Contact Owner (коефіцієнт -8 597).

Це означає, що розташування житла в Мумбаї в середньому підвищує орендну плату на ~19 210 одиниць порівняно з базовим містом, а контакт напряму з власником — знижує на ~8 597 порівняно з агентством.   

Загалом найсильніший вплив мають саме категоріальні ознаки (місто, тип контакту, меблювання), тоді як числові (Bathroom, Size) знаходяться нижче — це підтверджує, що One-Hot Encoding категоріальних змінних був важливим кроком для моделі.

In [56]:
# 5. Прогнози на тренувальній та тестовій вибірках.

y_train_pred = model.predict(X_train)                   # Прогнози на навчальній вибірці
y_test_pred = model.predict(X_test)                     # Прогнози на тестовій вибірці (нові дані!)

# Порівняння перших 10 прогнозів з реальністю
comparison = pd.DataFrame({
    'Реальна ціна': y_test.values[:10],
    'Прогноз': y_test_pred[:10].round(0).astype(int),
    'Помилка': (y_test.values[:10] - y_test_pred[:10]).round(0).astype(int)
})

print("Приклади прогнозів на тестовій вибірці:")
comparison

Приклади прогнозів на тестовій вибірці:


,Реальна ціна,Прогноз,Помилка
0,22000,27580,-5580
1,5000,3253,1747
2,37000,40243,-3243
3,8000,2725,5275
4,15000,15324,-324
5,20000,23256,-3256
6,8500,17716,-9216
7,7000,4289,2711
8,3000,47,2953
9,8000,6283,1717


## Завдання 6: Оцінка якості моделі (2 бали)

**Що потрібно зробити:**
1. Обчисліть MAE, RMSE та R² для навчальної та тестової вибірок
2. Порівняйте метрики та зробіть висновок про якість моделі
3. Проаналізуйте і дайте висновок, чи є ознаки перенавчання або недонавчання (**Нагадування**: перенавчання - коли модель дуже добре працює на тренувальних даних, але погано на тестових; недонавчання - коли модель погано працює навіть на тренувальних даних)
4. Побудуйте графік розсіювання "реальні vs прогнозовані значення" та зробіть висновок про якість моделі


In [58]:
# 1. MAE, RMSE та R² для навчальної та тестової вибірок

# Метрики для тренувальної вибірки
mae_train = mean_absolute_error(y_train, y_train_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

# Метрики для тестової вибірки
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

metrics_df = pd.DataFrame({
    'Вибірка': ['Train', 'Test'],
    'MAE': [mae_train, mae_test],
    'RMSE': [rmse_train, rmse_test],
    'R2': [r2_train, r2_test]
})

metrics_df.round(3)

,Вибірка,MAE,RMSE,R2
0,Train,5536.997,7657.984,0.693
1,Test,5540.000,7718.781,0.689


Метрики на тренувальній та тестовій вибірках майже не відрізняються, що свідчить про відсутність перенавчання. На тестових даних модель помиляється в середньому на 5 540 одиниць (MAE), при цьому RMSE (7 719) значно вищий за MAE — це вказує на наявність окремих випадків із суттєвими відхиленнями прогнозу.    

R² ≈ 0.689 означає, що лінійна регресія пояснює близько 69% варіації Rent — прийнятний результат для базової моделі, який можна покращити додатковими ознаками або складнішими алгоритмами.

In [60]:
# 4. Графік реальних та прогнозованих значень

fig = px.scatter(
    x=y_test,
    y=y_test_pred,
    title='Реальні vs прогнозовані значення орендної плати',
    labels={'x': 'Реальна ціна', 'y': 'Прогнозована ціна'},
    opacity=0.6
)

# Додаємо ідеальну лінію, де прогноз = реальність
max_val = max(y_test.max(), y_test_pred.max())

fig.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        name='Ідеальний прогноз',
        line=dict(color='red', dash='dash')
    )
)

fig.update_layout(height=500)
fig.show()

На графіку видно, що в нижньому ціновому діапазоні (до ~15K) точки досить щільно групуються навколо лінії ідеального прогнозу — тут модель працює найточніше.

Починаючи з ~20K розкид значно зростає.

Модель добре вгадує ціну дешевого житла — точки лежать близько до червоної лінії. Але чим дорожче житло, тим більше модель помиляється і занижує прогноз. Для покращення результатів у верхньому ціновому сегменті можна спробувати складніші моделі.

## Завдання 7: Аналіз помилок (4 бали)

**Що потрібно зробити:**
1. Обчисліть помилки (residuals = реальні - прогнозовані значення)
2. Побудуйте гістограму розподілу помилок
3. Створіть scatter plot помилок відносно величини прогнозованих значень. Чи росте помилка з ростом прогнозованого значення?
4. Знайдіть 5 прогнозів з найбільшими помилками
5. Проаналізуйте, на яких типах житла модель помиляється найбільше. Типи можна розрізняти за кількістю кімнат чи містом, наприклад.
6. Подумайте і напишіть, які наступні кроки ви б зробили, аби поліпшити якість моделі. Опціонально можна їх зробити і ми перевіримо :)

In [61]:
# 1. Розраховуємо помилки
residuals = y_test - y_test_pred

In [62]:
# 2. Гістограма помилок
fig = px.histogram(
    x=residuals,
    nbins=50,
    title='Розподіл помилок прогнозування',
    labels={'x': 'Помилка (реальні - прогнозовані)', 'count': 'Кількість'},
    color_discrete_sequence=['#e74c3c']
)
fig.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Помилка = 0")
fig.update_layout(height=400)
fig.show()

In [63]:
# 3. Scatter plot: помилки vs прогнозовані значення
fig = px.scatter(
    x=y_test_pred,
    y=residuals,
    title='Залежність помилок від прогнозованих значень',
    labels={
        'x': 'Прогнозована орендна плата',
        'y': 'Помилка (реальна - прогнозована)'
    },
    opacity=0.5
)

# Додаємо горизонтальну лінію на 0
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Без помилки")

fig.update_layout(height=400)
fig.show()

При низьких прогнозованих значеннях помилки компактні й близькі до нуля, але зі зростанням прогнозу розкид різко збільшується. Це означає, що модель стабільно працює для бюджетного та середнього сегменту, проте для дорожчого житла точність помітно падає.

In [65]:
# 4. Топ-5 прогнозів з найбільшими помилками

error_df = df_model.loc[y_test.index, [
    'Rent', 'BHK', 'Size', 'Bathroom', 'City',
    'Area Type', 'Furnishing Status', 'Tenant Preferred', 'Point of Contact'
]].copy()

error_df['Predicted_Rent'] = y_test_pred.round(0).astype(int)
error_df['Residual'] = (error_df['Rent'] - error_df['Predicted_Rent']).astype(int)
error_df['Abs_Error'] = error_df['Residual'].abs()
error_df['Error_%'] = (error_df['Abs_Error'] / error_df['Rent'] * 100).round(1)

top_errors = error_df.sort_values(by='Abs_Error', ascending=False).head(5)

top_errors

,Rent,BHK,Size,Bathroom,City,Area Type,Furnishing Status,Tenant Preferred,Point of Contact,Predicted_Rent,Residual,Abs_Error,Error_%
549,8000,2,650,2,Mumbai,Carpet Area,Semi-Furnished,Family,Contact Agent,40822,-32822,32822,410.3
3962,65000,3,1850,3,Hyderabad,Carpet Area,Furnished,Bachelors,Contact Owner,32195,32805,32805,50.5
904,8000,2,550,2,Mumbai,Carpet Area,Unfurnished,Family,Contact Agent,38855,-30855,30855,385.7
3520,65000,3,1444,3,Chennai,Super Area,Semi-Furnished,Bachelors,Contact Agent,35345,29655,29655,45.6
598,14000,2,570,2,Mumbai,Carpet Area,Furnished,Family,Contact Agent,43611,-29611,29611,211.5


In [69]:
# 5. На яких типах житла модель помиляється найбільше

for col in ['City', 'BHK']:
    print(f'\n--- Помилки по {col} ---')
    stats = (
        error_df.groupby(col)['Abs_Error']
        .agg(count='count', mean='mean', median='median')
        .sort_values('mean', ascending=False)
        .round(0)
    )
    display(stats)


--- Помилки по City ---


,count,mean,median
City,,,
Mumbai,118,9058.0,6831.0
Delhi,115,7234.0,5429.0
Hyderabad,165,5202.0,4524.0
Chennai,143,4733.0,3710.0
Kolkata,131,4178.0,2811.0
Bangalore,174,4044.0,2673.0



--- Помилки по BHK ---


,count,mean,median
BHK,,,
4,17,11579.0,8703.0
3,152,7800.0,6149.0
5,1,6266.0,6266.0
6,2,5306.0,5306.0
2,450,5076.0,3903.0
1,224,4478.0,3264.0


Аналіз помилок по групах показує, що модель найбільше помиляється для Mumbai та Delhi — міст із найрізноманітнішим ціновим діапазоном оренди. За кількістю кімнат найгірша точність у сегменті 4 BHK, тоді як результати для 5–6 BHK статистично ненадійні через малу кількість спостережень.

Аналіз помилок показав, що модель у більшості випадків прогнозує орендну плату досить точно — основна маса помилок знаходиться близько до нуля. Однак для дорожчого житла ситуація гірша: чим вища реальна ціна, тим сильніше модель помиляється і частіше занижує прогноз.   

Якщо подивитись на помилки по групах, найгірше модель працює для Мумбаї та Делі — це великі міста з дуже різноманітним ринком оренди, де ціни залежать від багатьох факторів. За кількістю кімнат найбільші помилки у сегменті 4 BHK, а для 5–6 BHK занадто мало даних, щоб робити надійні висновки.    

Причина цих помилок, скоріше за все, у тому, що модель не враховує деякі важливі ознаки — наприклад, конкретний район (Area Locality) та поверх (Floor), які були виключені через велику кількість унікальних значень. Для покращення якості можна спробувати закодувати ці ознаки іншим способом, застосувати логарифмування цільової змінної або використати нелінійні моделі.